# FoldMason Toolkit

This notebook demonstrates the two FoldMason tools:

1. **`foldmason-msa`** — multiple structure alignment (remote default; tuneable local mode)
2. **`foldmason-score-msa`** — score an existing MSA with average + per-column LDDT (local-only)

Reference: Gilchrist et al., *Science* 2026 ([DOI](https://doi.org/10.1126/science.ads6733)).

In [ ]:
import requests
from proto_tools.tools.structure_alignment import (
    FoldmasonMSAConfig, FoldmasonMSAInput, run_foldmason_msa,
    FoldmasonScoreMSAConfig, FoldmasonScoreMSAInput, run_foldmason_score_msa,
)

## Step 1: fetch a TIM-barrel structural family

We'll align three TIM-barrel triosephosphate isomerases from different organisms — a classic example of structurally similar enzymes spanning broad sequence divergence.

In [ ]:
tim_pdbs = {}
for pdb_id in ("1TIM", "8TIM", "1TPF"):
    tim_pdbs[pdb_id] = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30).text
structures = list(tim_pdbs.values())
ids = list(tim_pdbs.keys())
print(f"Fetched {len(structures)} TIM-barrel structures: {ids}")

## Step 2: align with `foldmason-msa` (remote)

Submits to the public FoldMason server at `search.foldseek.com/foldmason`. Returns the AA + 3Di alignments and the guide tree.

In [ ]:
msa_out = run_foldmason_msa(
    FoldmasonMSAInput(structures=structures, structure_ids=ids),
    FoldmasonMSAConfig(),
)
print(f"Aligned {msa_out.num_sequences} structures across {msa_out.alignment_length} columns")
print(f"Guide tree: {msa_out.newick_tree.strip()}")
print()
for line in msa_out.aa_msa_fasta.splitlines()[:6]:
    print(line[:100] + (" ..." if len(line) > 100 else ""))

## Step 3: score the alignment with `foldmason-score-msa` (local)

Computes per-column and average LDDT scores against the input structures. Useful as a quality metric or downstream design constraint. Local-only — `msa2lddt` is not exposed by the public server.

In [ ]:
score_out = run_foldmason_score_msa(
    FoldmasonScoreMSAInput(structures=structures, structure_ids=ids, aa_msa_fasta=msa_out.aa_msa_fasta),
    FoldmasonScoreMSAConfig(num_threads=2),
)
print(f"Average MSA LDDT: {score_out.average_lddt:.3f}")
print(f"Columns considered: {score_out.columns_considered}/{score_out.alignment_length}")
print(f"First 10 column scores: {[round(s, 2) for s in score_out.column_scores[:10]]}")